<a href="https://colab.research.google.com/github/leo5358/PL_1142/blob/main/HW4_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 整合 CVSS、EPSS、KEV 與 CWE 指標的資安漏洞 RAG 查詢系統

本 Notebook 依照 `proposal.md` 建立一個資安漏洞查詢系統雛形，示範如何整合 NVD、FIRST EPSS、CISA KEV 與 CWE 等資料，並透過向量資料庫與 RAG 回答自然語言查詢。

主要流程：

1. 建立漏洞資料欄位格式
2. 從公開 API 擷取 CVE / EPSS / KEV 資料
3. 將資料清理成統一表格
4. 將漏洞資料轉換成文字文件
5. 建立 Embedding 與向量資料庫
6. 使用 RAG 查詢漏洞風險與修補建議

> 若執行環境無法連網或尚未設定 API key，本 Notebook 也提供範例資料，可先跑通核心流程。

## 1. 安裝套件

第一次執行時請取消下面 cell 的註解並安裝需要的套件。若已安裝，可直接跳過。

In [1]:
!pip install pandas requests beautifulsoup4 faiss-cpu sentence-transformers gspread google-auth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 40.9 MB/s eta 0:00:00


## 2. 匯入套件與基本設定

In [2]:
import os
import re
import textwrap
from datetime import datetime

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 120)

NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"
EPSS_API_URL = "https://api.first.org/data/v1/epss"
CISA_KEV_URL = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1ge-RkD_jfq4NNu5egU2Fk48pBttx6vpIvsPWIh96B_o/edit?usp=sharing"
GOOGLE_WORKSHEET_NAME = "vul_db"
GOOGLE_WORKSHEET_GID = "548829531"

REQUIRED_COLUMNS = [
    "cve_id", "product", "summary", "cvss", "epss", "kev", "cwe",
    "affected_versions", "fixed_versions", "published_date", "references"
]

## 3. 建立範例漏洞資料

這份資料符合 proposal 中的 Google Sheets 欄位設計，可在尚未連接 API 或 Google Sheets 前先測試 RAG 流程。

In [3]:
sample_vulnerabilities = [
    {
        "cve_id": "CVE-2024-42008",
        "product": "Roundcube Webmail",
        "summary": "Cross-site scripting vulnerability in Roundcube Webmail that may allow script execution in a victim user's browser.",
        "cvss": 8.8,
        "epss": 0.53,
        "kev": "No",
        "cwe": "CWE-79",
        "affected_versions": "Before patched Roundcube release",
        "fixed_versions": "Upgrade to the latest maintained Roundcube version",
        "published_date": "2024-07-29",
        "references": "https://roundcube.net/news/",
    },
    {
        "cve_id": "CVE-2023-34362",
        "product": "MOVEit Transfer",
        "summary": "SQL injection vulnerability in MOVEit Transfer exploited in the wild to gain unauthorized access.",
        "cvss": 9.8,
        "epss": 0.94,
        "kev": "Yes",
        "cwe": "CWE-89",
        "affected_versions": "Multiple MOVEit Transfer versions",
        "fixed_versions": "Apply vendor security patches immediately",
        "published_date": "2023-06-02",
        "references": "https://www.cisa.gov/known-exploited-vulnerabilities-catalog",
    },
    {
        "cve_id": "CVE-2021-41773",
        "product": "Apache HTTP Server",
        "summary": "Path traversal and file disclosure vulnerability affecting Apache HTTP Server 2.4.49.",
        "cvss": 7.5,
        "epss": 0.97,
        "kev": "Yes",
        "cwe": "CWE-22",
        "affected_versions": "Apache HTTP Server 2.4.49",
        "fixed_versions": "Upgrade to Apache HTTP Server 2.4.51 or later",
        "published_date": "2021-10-05",
        "references": "https://httpd.apache.org/security/vulnerabilities_24.html",
    },
]

df = pd.DataFrame(sample_vulnerabilities)
df

,cve_id,product,summary,cvss,epss,kev,cwe,affected_versions,fixed_versions,published_date,references
0,CVE-2024-42008,Roundcube Webmail,Cross-site scripting vulnerability in Roundcube Webmail that may allow script execution in a victim user's browser.,8.8,0.53,No,CWE-79,Before patched Roundcube release,Upgrade to the latest maintained Roundcube version,2024-07-29,https://roundcube.net/news/
1,CVE-2023-34362,MOVEit Transfer,SQL injection vulnerability in MOVEit Transfer exploited in the wild to gain unauthorized access.,9.8,0.94,Yes,CWE-89,Multiple MOVEit Transfer versions,Apply vendor security patches immediately,2023-06-02,https://www.cisa.gov/known-exploited-vulnerabilities-catalog
2,CVE-2021-41773,Apache HTTP Server,Path traversal and file disclosure vulnerability affecting Apache HTTP Server 2.4.49.,7.5,0.97,Yes,CWE-22,Apache HTTP Server 2.4.49,Upgrade to Apache HTTP Server 2.4.51 or later,2021-10-05,https://httpd.apache.org/security/vulnerabilities_24.html


## 4. 從公開資料來源擷取資料

以下函式示範如何依 CVE 編號擷取 NVD、EPSS 與 CISA KEV 資料。NVD API 若有 API key，可設定環境變數 `NVD_API_KEY` 提高請求額度。

In [13]:
def fetch_nvd_cve(cve_id: str) -> dict:
    """Fetch basic CVE metadata from NVD."""
    headers = {}
    if os.getenv("NVD_API_KEY"):
        headers["apiKey"] = os.getenv("NVD_API_KEY")

    response = requests.get(NVD_API_URL, params={"cveId": cve_id}, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()

    vulnerabilities = data.get("vulnerabilities", [])
    if not vulnerabilities:
        return {}

    cve = vulnerabilities[0].get("cve", {})
    descriptions = cve.get("descriptions", [])
    english_description = next((d["value"] for d in descriptions if d.get("lang") == "en"), "")

    metrics = cve.get("metrics", {})
    cvss = None
    for metric_key in ["cvssMetricV31", "cvssMetricV30", "cvssMetricV2"]:
        if metrics.get(metric_key):
            cvss = metrics[metric_key][0].get("cvssData", {}).get("baseScore")
            break

    cwe = ""
    weaknesses = cve.get("weaknesses", [])
    if weaknesses:
        cwe_descriptions = weaknesses[0].get("description", [])
        cwe = next((d["value"] for d in cwe_descriptions if d.get("lang") == "en"), "")

    # Fix: references in NVD API v2 is directly a list of objects
    reference_list = cve.get("references", [])
    references = "; ".join(ref.get("url", "") for ref in reference_list[:5])

    return {
        "cve_id": cve_id,
        "product": "",
        "summary": english_description,
        "cvss": cvss,
        "epss": None,
        "kev": "No",
        "cwe": cwe,
        "affected_versions": "",
        "fixed_versions": "",
        "published_date": cve.get("published", "")[:10],
        "references": references,
    }

## 5. 資料清理與風險分級

將 CVSS、EPSS、KEV 等指標整理成較容易分析的欄位。

In [5]:
def cvss_severity(score) -> str:
    if pd.isna(score):
        return "Unknown"
    score = float(score)
    if score >= 9.0:
        return "Critical"
    if score >= 7.0:
        return "High"
    if score >= 4.0:
        return "Medium"
    if score > 0:
        return "Low"
    return "None"


def epss_percent(score) -> str:
    if pd.isna(score):
        return "Unknown"
    return f"{float(score) * 100:.1f}%"


def normalize_vulnerability_table(vuln_df: pd.DataFrame) -> pd.DataFrame:
    normalized = vuln_df.copy()
    for column in REQUIRED_COLUMNS:
        if column not in normalized.columns:
            normalized[column] = ""

    normalized["cvss_severity"] = normalized["cvss"].apply(cvss_severity)
    normalized["epss_percent"] = normalized["epss"].apply(epss_percent)
    normalized["risk_priority"] = normalized.apply(
        lambda row: "Immediate" if row["kev"] == "Yes" or float(row["epss"] or 0) >= 0.8 else "Normal",
        axis=1,
    )
    return normalized[REQUIRED_COLUMNS + ["cvss_severity", "epss_percent", "risk_priority"]]


df = normalize_vulnerability_table(df)
df

,cve_id,product,summary,cvss,epss,kev,cwe,affected_versions,fixed_versions,published_date,references,cvss_severity,epss_percent,risk_priority
0,CVE-2024-42008,Roundcube Webmail,Cross-site scripting vulnerability in Roundcube Webmail that may allow script execution in a victim user's browser.,8.8,0.53,No,CWE-79,Before patched Roundcube release,Upgrade to the latest maintained Roundcube version,2024-07-29,https://roundcube.net/news/,High,53.0%,Normal
1,CVE-2023-34362,MOVEit Transfer,SQL injection vulnerability in MOVEit Transfer exploited in the wild to gain unauthorized access.,9.8,0.94,Yes,CWE-89,Multiple MOVEit Transfer versions,Apply vendor security patches immediately,2023-06-02,https://www.cisa.gov/known-exploited-vulnerabilities-catalog,Critical,94.0%,Immediate
2,CVE-2021-41773,Apache HTTP Server,Path traversal and file disclosure vulnerability affecting Apache HTTP Server 2.4.49.,7.5,0.97,Yes,CWE-22,Apache HTTP Server 2.4.49,Upgrade to Apache HTTP Server 2.4.51 or later,2021-10-05,https://httpd.apache.org/security/vulnerabilities_24.html,High,97.0%,Immediate


## 6. 將爬蟲資料存入 Google Sheets

正式專題可使用 Google Sheets 管理爬蟲/API 擷取後的資料。流程建議為：

1. 使用 `build_dataset_from_cves()` 從 NVD、EPSS、KEV 建立資料表
2. 使用 `upload_to_google_sheets()` 寫入 Google Sheets
3. 後續 RAG 流程使用 `load_from_google_sheets()` 從 Google Sheets 讀回資料

使用前需先建立 Google Cloud Service Account，下載憑證 JSON，並將試算表分享給該 Service Account email。

In [15]:
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd

# 1. 執行 Colab 驗證 (這會跳出授權視窗)
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. 改寫取得工作表的函式，直接使用全域的 gc 客戶端
def get_google_worksheet(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    """Authorize Google Sheets via Colab and return the target worksheet."""
    spreadsheet = gc.open_by_url(spreadsheet_url)
    return spreadsheet.worksheet(worksheet_name)

def ensure_google_sheet_schema(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
    required_columns: list[str] = REQUIRED_COLUMNS,
):
    """Check the worksheet header and rewrite it if columns do not match."""
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    rows = worksheet.get_all_values()

    if not rows:
        worksheet.update([required_columns], "A1")
        return {"status": "created_header", "columns": required_columns}

    current_columns = rows[0]
    if current_columns == required_columns:
        return {"status": "ok", "columns": current_columns}

    records = worksheet.get_all_records()
    fixed_df = pd.DataFrame(records)
    for column in required_columns:
        if column not in fixed_df.columns:
            fixed_df[column] = ""
    fixed_df = fixed_df[required_columns]

    worksheet.clear()
    worksheet.update([required_columns] + fixed_df.fillna("").values.tolist())
    return {"status": "fixed", "before": current_columns, "after": required_columns}

def upload_to_google_sheets(
    vuln_df: pd.DataFrame,
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    """Overwrite Google Sheets with vulnerability data."""
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    output_df = vuln_df.copy()
    for column in REQUIRED_COLUMNS:
        if column not in output_df.columns:
            output_df[column] = ""
    output_df = output_df[REQUIRED_COLUMNS]
    worksheet.clear()
    worksheet.update([output_df.columns.tolist()] + output_df.fillna("").values.tolist())

def append_to_google_sheets(
    vuln_df: pd.DataFrame,
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
):
    """Append newly crawled rows to Google Sheets without clearing old data."""
    ensure_google_sheet_schema(spreadsheet_url, worksheet_name)
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    output_df = vuln_df.copy()
    for column in REQUIRED_COLUMNS:
        if column not in output_df.columns:
            output_df[column] = ""
    worksheet.append_rows(output_df[REQUIRED_COLUMNS].fillna("").values.tolist(), value_input_option="USER_ENTERED")

def load_from_google_sheets(
    spreadsheet_url: str = GOOGLE_SHEET_URL,
    worksheet_name: str = GOOGLE_WORKSHEET_NAME,
    fix_schema: bool = True,
) -> pd.DataFrame:
    """Load vulnerability data from Google Sheets into a DataFrame."""
    if fix_schema:
        ensure_google_sheet_schema(spreadsheet_url, worksheet_name)
    worksheet = get_google_worksheet(spreadsheet_url, worksheet_name)
    records = worksheet.get_all_records()
    return pd.DataFrame(records, columns=REQUIRED_COLUMNS)

# 測試執行區塊
try:
    # 這裡直接拿你前面做好的範例 df 來測試寫入
    print("正在將資料寫入 Google Sheets...")
    upload_to_google_sheets(df)

    # 範例：從 vul_db 讀回資料
    schema_result = ensure_google_sheet_schema()
    print(f"Sheet Schema Check: {schema_result}")

    loaded_df = load_from_google_sheets()
    print("成功從 Google Sheets 讀取資料，共", len(loaded_df), "筆")

except Exception as e:
    print(f"執行時發生錯誤: {e}")

正在將資料寫入 Google Sheets...
Sheet Schema Check: {'status': 'ok', 'columns': ['cve_id', 'product', 'summary', 'cvss', 'epss', 'kev', 'cwe', 'affected_versions', 'fixed_versions', 'published_date', 'references']}
成功從 Google Sheets 讀取資料，共 3 筆


## 7. 將漏洞資料轉換為 RAG 文件

每一筆漏洞會被轉成一段文字，再交給 embedding model 建立向量。

In [7]:
def vulnerability_to_document(row: pd.Series) -> str:
    return textwrap.dedent(f"""
    CVE ID: {row['cve_id']}
    Product: {row['product']}
    Summary: {row['summary']}
    CVSS Score: {row['cvss']} ({row['cvss_severity']})
    EPSS Score: {row['epss']} ({row['epss_percent']})
    CISA KEV: {row['kev']}
    CWE: {row['cwe']}
    Affected Versions: {row['affected_versions']}
    Fixed Versions: {row['fixed_versions']}
    Published Date: {row['published_date']}
    Risk Priority: {row['risk_priority']}
    Mitigation: {row['fixed_versions']}
    References: {row['references']}
    """).strip()


documents = [vulnerability_to_document(row) for _, row in df.iterrows()]
ids = df["cve_id"].tolist()
metadatas = df[["cve_id", "product", "cvss_severity", "kev", "cwe", "risk_priority"]].to_dict("records")

print(documents[0])

CVE ID: CVE-2024-42008
Product: Roundcube Webmail
Summary: Cross-site scripting vulnerability in Roundcube Webmail that may allow script execution in a victim user's browser.
CVSS Score: 8.8 (High)
EPSS Score: 0.53 (53.0%)
CISA KEV: No
CWE: CWE-79
Affected Versions: Before patched Roundcube release
Fixed Versions: Upgrade to the latest maintained Roundcube version
Published Date: 2024-07-29
Risk Priority: Normal
Mitigation: Upgrade to the latest maintained Roundcube version
References: https://roundcube.net/news/


## 8. 建立向量資料庫

使用 FAISS 儲存向量索引，並使用 Sentence Transformers 產生 embedding。FAISS 只負責近似向量搜尋，不會保存原始文字與 metadata，所以這裡用 `FaissRetriever` 類別同時管理索引、文件、CVE ID 與 metadata。

若環境尚未安裝相關套件，會改用簡易關鍵字檢索 fallback，方便先測試查詢流程。

In [8]:
class KeywordRetriever:
    """Simple fallback retriever when vector database packages are unavailable."""

    def __init__(self, docs: list[str], ids: list[str], metadatas: list[dict]):
        self.docs = docs
        self.ids = ids
        self.metadatas = metadatas

    @staticmethod
    def tokenize(text: str) -> set[str]:
        return set(re.findall(r"[A-Za-z0-9_-]+", text.lower()))

    def query(self, query_text: str, top_k: int = 3) -> list[dict]:
        query_tokens = self.tokenize(query_text)
        scored = []
        for doc_id, doc, metadata in zip(self.ids, self.docs, self.metadatas):
            doc_tokens = self.tokenize(doc)
            score = len(query_tokens & doc_tokens)
            scored.append({"id": doc_id, "document": doc, "metadata": metadata, "score": score})
        return sorted(scored, key=lambda item: item["score"], reverse=True)[:top_k]


class FaissRetriever:
    """FAISS vector retriever backed by Sentence Transformers embeddings."""

    def __init__(self, docs: list[str], ids: list[str], metadatas: list[dict], model_name: str = "BAAI/bge-small-en-v1.5"):
        import faiss
        import numpy as np
        from sentence_transformers import SentenceTransformer

        self.docs = docs
        self.ids = ids
        self.metadatas = metadatas
        self.model = SentenceTransformer(model_name)

        embeddings = self.model.encode(docs, normalize_embeddings=True, show_progress_bar=False)
        embeddings = np.asarray(embeddings, dtype="float32")

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

    def query(self, query_text: str, top_k: int = 3) -> list[dict]:
        import numpy as np

        query_embedding = self.model.encode([query_text], normalize_embeddings=True, show_progress_bar=False)
        query_embedding = np.asarray(query_embedding, dtype="float32")
        scores, indices = self.index.search(query_embedding, min(top_k, len(self.docs)))

        results = []
        for score, index in zip(scores[0], indices[0]):
            if index == -1:
                continue
            results.append({
                "id": self.ids[index],
                "document": self.docs[index],
                "metadata": self.metadatas[index],
                "score": float(score),
            })
        return results

    def save_index(self, path: str = "vulnerability.faiss"):
        import faiss

        faiss.write_index(self.index, path)


def build_retriever(documents: list[str], ids: list[str], metadatas: list[dict]):
    try:
        return FaissRetriever(documents, ids, metadatas)
    except Exception as exc:
        print(f"FAISS vector stack unavailable, using keyword fallback: {exc}")
        return KeywordRetriever(documents, ids, metadatas)


retriever = build_retriever(documents, ids, metadatas)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 9. RAG 檢索函式

In [9]:
def retrieve_context(query: str, top_k: int = 3) -> list[dict]:
    if isinstance(retriever, (KeywordRetriever, FaissRetriever)):
        return retriever.query(query, top_k=top_k)
    raise TypeError(f"Unsupported retriever type: {type(retriever)}")


retrieve_context("哪些漏洞已列入 KEV？", top_k=2)

[{'id': 'CVE-2023-34362',
  'document': 'CVE ID: CVE-2023-34362\nProduct: MOVEit Transfer\nSummary: SQL injection vulnerability in MOVEit Transfer exploited in the wild to gain unauthorized access.\nCVSS Score: 9.8 (Critical)\nEPSS Score: 0.94 (94.0%)\nCISA KEV: Yes\nCWE: CWE-89\nAffected Versions: Multiple MOVEit Transfer versions\nFixed Versions: Apply vendor security patches immediately\nPublished Date: 2023-06-02\nRisk Priority: Immediate\nMitigation: Apply vendor security patches immediately\nReferences: https://www.cisa.gov/known-exploited-vulnerabilities-catalog',
  'metadata': {'cve_id': 'CVE-2023-34362',
   'product': 'MOVEit Transfer',
   'cvss_severity': 'Critical',
   'kev': 'Yes',
   'cwe': 'CWE-89',
   'risk_priority': 'Immediate'},
  'score': 0.6224347949028015},
 {'id': 'CVE-2024-42008',
  'document': "CVE ID: CVE-2024-42008\nProduct: Roundcube Webmail\nSummary: Cross-site scripting vulnerability in Roundcube Webmail that may allow script execution in a victim user's br

## 10. 產生查詢結果

這個版本不使用 AI 產生回答。使用者輸入查詢後，系統只做 FAISS/關鍵字檢索，並把最相關的漏洞資料整理成固定格式輸出。

In [10]:
def answer_with_template(query: str, contexts: list[dict]) -> str:
    if not contexts:
        return "找不到相關漏洞資料。"

    lines = [f"查詢問題：{query}", "", "檢索到的相關漏洞："]
    for item in contexts:
        doc = item["document"]
        wanted_prefixes = [
            "CVE ID:", "Product:", "CVSS Score:", "EPSS Score:", "CISA KEV:",
            "CWE:", "Affected Versions:", "Fixed Versions:", "Mitigation:", "References:"
        ]
        extracted = [line.strip() for line in doc.splitlines() if any(line.strip().startswith(prefix) for prefix in wanted_prefixes)]
        lines.append("\n".join(extracted))
        lines.append("")
    return "\n".join(lines).strip()


def search_vulnerabilities(query: str, top_k: int = 3) -> str:
    contexts = retrieve_context(query, top_k=top_k)
    return answer_with_template(query, contexts)

## 11. 輸入查詢

In [11]:
query = input("請輸入漏洞查詢，例如：哪些漏洞已列入 KEV？")
print(search_vulnerabilities(query, top_k=3))

請輸入漏洞查詢，例如：哪些漏洞已列入 KEV？copy fail
查詢問題：copy fail

檢索到的相關漏洞：
CVE ID: CVE-2023-34362
Product: MOVEit Transfer
CVSS Score: 9.8 (Critical)
EPSS Score: 0.94 (94.0%)
CISA KEV: Yes
CWE: CWE-89
Affected Versions: Multiple MOVEit Transfer versions
Fixed Versions: Apply vendor security patches immediately
Mitigation: Apply vendor security patches immediately
References: https://www.cisa.gov/known-exploited-vulnerabilities-catalog

CVE ID: CVE-2024-42008
Product: Roundcube Webmail
CVSS Score: 8.8 (High)
EPSS Score: 0.53 (53.0%)
CISA KEV: No
CWE: CWE-79
Affected Versions: Before patched Roundcube release
Fixed Versions: Upgrade to the latest maintained Roundcube version
Mitigation: Upgrade to the latest maintained Roundcube version
References: https://roundcube.net/news/

CVE ID: CVE-2021-41773
Product: Apache HTTP Server
CVSS Score: 7.5 (High)
EPSS Score: 0.97 (97.0%)
CISA KEV: Yes
CWE: CWE-22
Affected Versions: Apache HTTP Server 2.4.49
Fixed Versions: Upgrade to Apache HTTP Server 2.4.51 or late

## 12. 後續可擴充方向

1. 將 CVE 清單改由 Google Sheets 讀取，爬蟲/API 更新後再寫回 Google Sheets。
2. 加入產品名稱與版本解析，提升受影響版本與修補版本的準確度。
3. 加入 CWE 名稱對照表，例如 CWE-79 對應 Cross-site Scripting。
4. 使用排程工具定期同步 NVD、EPSS 與 KEV。
5. 建立 Streamlit 或 Gradio 介面，提供自然語言查詢頁面。
6. 在回答中加入引用來源與信心分數，降低 RAG 幻覺風險。